## 1 & 2. Load Dataset & Build CNN

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os

# Constants
IMG_SIZE = (128, 128)
BATCH_SIZE = 8


train_ds = tf.keras.utils.image_dataset_from_directory(
    'fashion_dataset',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='training',
    seed=123
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'fashion_dataset',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='validation',
    seed=123
)

class_names = train_ds.class_names
print("Categories found:", class_names)

# Build the CNN model with requested stack
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128, 128, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

Found 30 files belonging to 3 classes.
Using 24 files for training.
Found 30 files belonging to 3 classes.
Using 6 files for validation.
Categories found: ['bag', 'shoe', 't_shirt']


C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## 3. Train for 5 epochs

In [2]:
print("\n--- Training Model ---")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


--- Training Model ---
Epoch 1/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 375ms/step - accuracy: 0.3750 - loss: 5.9464 - val_accuracy: 0.8333 - val_loss: 3.4338
Epoch 2/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 219ms/step - accuracy: 0.5833 - loss: 7.1471 - val_accuracy: 0.8333 - val_loss: 2.9249
Epoch 3/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 206ms/step - accuracy: 0.6250 - loss: 4.5487 - val_accuracy: 0.8333 - val_loss: 0.6952
Epoch 4/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 210ms/step - accuracy: 0.9167 - loss: 0.4860 - val_accuracy: 0.3333 - val_loss: 1.0358
Epoch 5/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 204ms/step - accuracy: 0.7500 - loss: 0.6739 - val_accuracy: 0.1667 - val_loss: 1.7294


## 4. Evaluate on Unseen Test Set & Find Misclassifications

In [3]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    'test_dataset',
    image_size=IMG_SIZE,
    batch_size=1,
    shuffle=False
)

print("\n--- Evaluating Test Set & Checking Misclassifications ---")
misclassified_count = 0

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    pred_label = np.argmax(preds[0])
    true_label = labels[0].numpy()
    
    if pred_label != true_label:
        misclassified_count += 1
        print(f"Misclassified! Predicted: {class_names[pred_label]}, Actual: {class_names[true_label]}")

print(f"\nTotal test evaluation complete. Misclassified images: {misclassified_count}")

Found 9 files belonging to 3 classes.

--- Evaluating Test Set & Checking Misclassifications ---
Misclassified! Predicted: t_shirt, Actual: shoe
Misclassified! Predicted: t_shirt, Actual: shoe
Misclassified! Predicted: t_shirt, Actual: shoe

Total test evaluation complete. Misclassified images: 3
